# Valuation and Hedging of Options
In this project, we price and hedge European and American options using different methods.
We select Apple Inc. (AAPL) as the underlying asset and impleemnt the Black-Scholes model, Monte-Carlo simulations under Geometric Brownian Motion, and the Binomial model for American options.

All formulas are implemented manually in Python as required.

In [11]:
import numpy as np
import yfinance as yf
import pandas as pd
import math

## Config

* Period: 01-01-2024 -- 01-01-2025 (1 year).
* For the risk free rate we decide to use 1Y US treasury yield at that time (approx. 5%). **Source: https://fred.stlouisfed.org/series/DGS1**
* Assume strike (K) as the mean price of the asset

In [15]:
ticker="AAPL"
df=yf.download(ticker, start="2024-01-01", end="2025-01-01")

S=df["Close"].dropna()
r=0.05 #risk free rate (1Y )
T=1 #time to maturity (1Y)
log_returns=np.log(S/S.shift(1)).dropna()
sigma=log_returns.std()*np.sqrt(252) #annualized volatility
K=S.mean()
S0=S.iloc[-1] #current stock price



[*********************100%***********************]  1 of 1 completed


## Black-Scholes model

The Black-Scholes-Merton equation provides a framwork for option pricing. We denote the stock price as $S_t$ , $K$ be the strike price, $T$ be the time, $V$ - option price value, $r$ - risk free rate and $sigma$ - volatility(constant). The equation serves as a (foundational) tool for pricing options in continuous time, it is universally use to determine fair value of **European options**, including calls and puts, on various underlying assets like stocks, indices, and commodities.

The model assumes:
- Lognormal distribution of prices
- Constant volatility
- Constant risk-free rate
- No dividends
- No arbitrage
- Continuous time

Black-Scholes equation:

$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S_t^2 \frac{\partial^2 V}{\partial S_t^2} + rS_t \frac{\partial V}{\partial S_t} - rV = 0, \quad with \quad 0<S_t<S_T, \quad 0 \leq t<T$

In [17]:
#helpers d1 and d2
def d1(S, K, r, sigma, T):
    return (np.log(S/K)+(r+0.5*sigma**2)*T) / (sigma*np.sqrt(T))

def d2(S, K, r, sigma, T):
    return d1(S, K, r, sigma, T) - sigma*np.sqrt(T)


Call Price is determined by the max payoff it could generate. Indeed, if $S_t